# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rslns/FlyRankAI_A1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [6]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF_TOKEN loaded:", bool(HF_TOKEN))

HF_TOKEN loaded: True


In [7]:
from huggingface_hub import hf_hub_download
import pyarrow.parquet as pq

march_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

march_table = pq.read_table(march_file)

print(march_table.num_rows, "rows")
print(march_table.num_columns, "columns")

9841378 rows
31 columns


In [8]:
march_df = march_table.to_pandas()

print("March shape:", march_df.shape)
print(
    "Date range:",
    march_df["report_date"].min(),
    "to",
    march_df["report_date"].max()
)

march_df.head()

March shape: (9841378, 31)
Date range: 2026-03-01 to 2026-03-31


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,None,20,0,67,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,None,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,None,125,1,616,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,None,7,0,28,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,None,11,0,25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*
## Unit of analysis + time window

**Unit of analysis:** One row represents the daily performance of one content item for one client on one report date.

**Time window:** I will use a mid-panel month, **March 2026**, for development. This avoids using the final June 2026 month as the development window.


In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*
## Fields: feature / label / context / excluded

**Features**

* `gsc_impressions` — known from the search-performance window available before the decision.
* `gsc_clicks` — known from the search-performance window available before the decision.
* `gsc_avg_position` — known from the search-performance window available before the decision.
* `ga4_sessions` — known from the analytics window available before the decision.
* `ga4_engaged_sessions` — known from the analytics window available before the decision.

**Label / proxy**

* `gsc_clicks` in a later outcome window — the observed outcome I would use to judge whether a content item needs attention. The future outcome must not be used as a feature.

**Context**

* `client_hash_id` — used to identify/group clients, not as a model feature.
* `content_hash_id` — used to identify content items, not as a model feature.
* `report_date` — used to define the time window, not as a model feature.

**Excluded**

* `gsc_avg_position` from the future outcome window — excluded because it would contain future information.
* `gsc_clicks` from the future outcome window when building features — excluded for the same reason.
* `trend_direction` / `trend_pct` — excluded because they are derived from the outcome/trend definition and could leak label information.
* `fact_content_daily_performance_sample` — excluded for development because it represents the final June 2026 month and should remain a sealed test window.


In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*
### Five features

For the development decision, I will use these five features:

1. `gsc_impressions` — available from the search-performance data before the decision moment.
2. `gsc_clicks` — available from the search-performance data before the decision moment.
3. `gsc_avg_position` — available from the search-performance data before the decision moment.
4. `ga4_sessions` — available from the analytics data before the decision moment.
5. `ga4_engaged_sessions` — available from the analytics data before the decision moment.

These features describe observed search and engagement performance that is available at the time of the decision.


### Deliberate leakage check

For the leakage demonstration, I define an observed outcome as whether the content item received at least one GSC click. I then deliberately include that outcome itself as a feature. This is leakage because the model is given the answer it is supposed to predict. The resulting score should be artificially perfect or near-perfect. I remove the leaked column afterward and keep only the honest features.


In [11]:
# Create an observed outcome
march_df["clicked"] = (march_df["gsc_clicks"] > 0).astype(int)

# Deliberately leak the outcome into the features
leaky_features = march_df[
    [
        "gsc_impressions",
        "gsc_avg_position",
        "ga4_sessions",
        "ga4_engaged_sessions",
        "clicked"
    ]
].copy()

print("Leaky feature columns:")
print(leaky_features.columns.tolist())

Leaky feature columns:
['gsc_impressions', 'gsc_avg_position', 'ga4_sessions', 'ga4_engaged_sessions', 'clicked']


In [12]:
honest_features = leaky_features.drop(columns=["clicked"])

print("Honest feature columns:")
print(honest_features.columns.tolist())

Honest feature columns:
['gsc_impressions', 'gsc_avg_position', 'ga4_sessions', 'ga4_engaged_sessions']


In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*
This March 2026 slice has uneven data availability. Only 3,611,061 of 9,841,378 rows have GSC data available, so unavailable GSC data cannot be treated as zero performance. Client history also differs, so this one-month slice cannot describe every client's full search history. It is suitable for development, but it cannot by itself tell us what will happen in a future month.


In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.